# 02 - Transformação

## Pipeline de Transformação e Engenharia de Atributos (02_transform)

### Visão Geral e Arquitetura do Pipeline

Este notebook implementa a camada de transformação (Silver) e a construção das Tabelas Analíticas de Modelagem (ABT - Gold). O objetivo é processar os dados brutos de crédito, consolidar a variável resposta e estruturar os dados para os experimentos de modelagem no notebook 03_modeling.

O pipeline opera inteiramente em memória para transformações intermediárias, persistindo em disco exclusivamente as bases consolidadas para modelagem.

---

### Fases do Pipeline

1. **Higienização e Tipagem Estrutural (Camada Silver):**
   - cadastral_tratada: relação 1:1 por cliente com consolidação por recência e fallback.
   - emprestimos_tratada: relação 1:N por contrato, com conservação volumétrica integral.
   - parcelas_tratada: relação 1:M por prestação, com remoção estrita de duplicatas exatas.

2. **Rollup de Parcelas e Construção do Target:**
   - Agrupamento em memória gerando o DataFrame parcelas_por_contrato (grão de contrato).
   - Filtragem pela versão mais recente da parcela e consolidação de pagamentos fracionados (tolerância de R$ 0,01).
   - Apuração do horizonte de observação MOB 12 via tempo corrido (data de decisão do contrato + 360 dias).
   - Critério de maturação de safra referenciado no limite empírico observável da carteira: max(data_real_pagamento).
   - Construção do target OVER60MOB12 (1 para atraso superior a 60 dias dentro do MOB 12; 0 para contratos maduros sem atraso crítico; NA para safras imaturas ou com incerteza).

3. **ABT de Baseline (Benchmark sem Engenharia):**
   - Merge estrutural direto das bases tratadas com a target.
   - Sem criação de novas features derivadas.
   - Sem encoding, escalonamento ou imputações manuais.
   - Persistência em disco: **dados/abt_baseline.parquet**.

4. **Engenharia de Atributos (Feature Engineering):**
   - Features comportamentais históricas calculadas nas bases individuais (rollups por cliente).
   - Merge analítico completo.
   - Features de interação cruzadas pós-merge (idade na decisão, comprometimento de renda, razões financeiras).

5. **ABT Final para Modelagem:**
   - Validações de integridade, ausência de leakage e tipagem final.
   - Persistência em disco: dados/abt_modelagem.parquet.

6. **Preparação da Base de Inferência (Submissão):**
   - Aplicação estrita dos mesmos tratamentos cadastrais da base de treino.
   - Consolidação das features comportamentais históricas para a data de solicitação.
   - Persistência da base de escoragem para predição em produção.

---

### Diretrizes de Engenharia e Boas Práticas

- **Prevenção de Vazamento Temporal (Data Leakage):** Apenas informações disponíveis no momento da tomada de decisão (data_decisao no histórico e data_solicitacao na submissão) podem ser utilizadas como preditores. Variáveis de desfecho do contrato não entram como features.
- **Validação por Invariantes:** Asserções de pipeline (assert) devem testar consistência lógica e conservação volumétrica dinâmica entre entradas e saídas, evitando valores fixos (magic numbers).
- **Critério de Maturação da Safra (Target):**
  - **Evento Mau (1):** Contratos com atraso superior a 60 dias até a 12ª parcela.
  - **Evento Bom (0):** Contratos maduros (pelo menos 12 parcelas observadas ou com mais de 365 dias de histórico) sem atraso superior a 60 dias.
  - **Indeterminado (NA):** Contratos imaturos sem atraso crítico (excluídos do treinamento supervisionado).

In [ ]:
import sys
sys.path.append("..")

import src.functions as fn #Funcoes criadas no projeto
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Carrega cópias de trabalho mantendo os arquivos originais intactos.
cadastral = pd.read_parquet("../dados/base_cadastral.parquet").copy()
submissao = pd.read_parquet("../dados/base_submissao.parquet").copy()
emprestimos = pd.read_parquet("../dados/historico_emprestimos.parquet").copy()
parcelas = pd.read_parquet("../dados/historico_parcelas.parquet").copy()

print(
    f"cadastral: {cadastral.shape} | submissao: {submissao.shape} | "
    f"emprestimos: {emprestimos.shape} | parcelas: {parcelas.shape}"
)

## Tratamento de Dados para as tabelas-base de treino

In [ ]:
# --- TRATAMENTO E CONSOLIDAÇÃO: BASE CADASTRAL ---

# 1. Guarda referências dinâmicas para asserções
n_clientes_esperados = cadastral["id_cliente"].nunique()

# 2. Identificação de datas de nascimento sentinela e conversão de tipos
datas_convertidas = pd.to_datetime(cadastral["data_nascimento"], errors="coerce")
cadastral["data_nascimento_invalida"] = (
    cadastral["data_nascimento"].notna() & (datas_convertidas.dt.year <= 1920)
)
cadastral.loc[cadastral["data_nascimento_invalida"], "data_nascimento"] = None

# Aplica conversão padronizada de tipos estruturais
cadastral, rel_tipos = fn.converter_tipos(
    cadastral,
    datas={"data_nascimento": ["%Y-%m-%d", "%d/%m/%Y"]},
    numericas_texto={"renda_anual": {"decimal": ",", "milhar": "."}} if cadastral["renda_anual"].dtype == "object" else None,
    inteiros=["qtd_membros_familia"],
)

# 3. Padronização de variáveis categóricas e conversão de placeholders em NaN
cols_categoricas = [
    c for c in cadastral.select_dtypes(include=["object", "string", "category"]).columns 
    if c not in ["data_nascimento"]
]
cadastral, rel_cat = fn.padronizar_categoricas(cadastral, colunas=cols_categoricas)

# 4. Consolidação de duplicatas: recência com preenchimento de nulos
mascara_dup = cadastral.duplicated(subset=["id_cliente"], keep=False)
cad_unicos = cadastral.loc[~mascara_dup].set_index("id_cliente")

cad_1a = cadastral.loc[cadastral.duplicated(subset=["id_cliente"], keep="last")].set_index("id_cliente")
cad_2a = cadastral.loc[cadastral.duplicated(subset=["id_cliente"], keep="first")].set_index("id_cliente")

# A 2ª ocorrência prevalece; onde houver NaN, herda o valor da 1ª ocorrência
cad_dups_consolidadas = cad_2a.combine_first(cad_1a)

cadastral_tratada = (
    pd.concat([cad_unicos, cad_dups_consolidadas])
    .sort_index()
    .reset_index()
)

# 5. Otimização de tipos finais para controle de memória
cadastral_tratada["id_cliente"] = cadastral_tratada["id_cliente"].astype("int64")
cadastral_tratada["qtd_filhos"] = cadastral_tratada["qtd_filhos"].astype("Int64")
for col in cols_categoricas:
    cadastral_tratada[col] = cadastral_tratada[col].astype("category")

# 6. Remoção de colunas sensíveis/não utilizadas no modelo
cols_para_remover = ["sexo"]
n_cols_antes_drop = cadastral_tratada.shape[1]
cadastral_tratada = cadastral_tratada.drop(columns=cols_para_remover)

# 7. Asserções para validação do pipeline
assert cadastral_tratada.shape[0] == n_clientes_esperados, (
    f"Esperado {n_clientes_esperados} linhas, obtido {cadastral_tratada.shape[0]}"
)
assert cadastral_tratada["id_cliente"].nunique() == n_clientes_esperados, (
    "Falha: id_cliente não é único na base consolidada"
)
assert cadastral_tratada.shape[1] == (n_cols_antes_drop - len(cols_para_remover)), (
    "Inconsistência no número de colunas após exclusão de campos indesejados"
)

print("--- Base Cadastral Tratada com Sucesso ---")
print(f"Shape final: {cadastral_tratada.shape}")
print(f"Clientes únicos garantidos: {cadastral_tratada['id_cliente'].nunique()}")
print(f"Total com data_nascimento_invalida: {cadastral_tratada['data_nascimento_invalida'].sum()}")

In [ ]:
cadastral_tratada.info()

In [ ]:
cadastral_tratada.head()

In [ ]:
# --- TRATAMENTO E TIPAGEM: BASE EMPRESTIMOS ---

# 1. Guarda referências dinâmicas da base de entrada
n_linhas_esperadas_emp = len(emprestimos)
n_contratos_esperados_emp = emprestimos["id_contrato"].nunique()

emprestimos_proc = emprestimos.copy()

# 2. Harmonização semântica de colunas (alinha com submissao e evita data leakage)
mapa_renomeacao_emp = {
    "valor_credito": "valor_credito_concedido",
    "valor_solicitado": "valor_credito",
}
emprestimos_proc = emprestimos_proc.rename(columns=mapa_renomeacao_emp)

# 3. Mapeamento explícito de todas as colunas temporais presentes
formatos_data = ["%Y-%m-%d", "%d/%m/%Y"]
cols_datas_emp = [c for c in emprestimos_proc.columns if c.startswith("data_")]
datas_dict_emp = {c: formatos_data for c in cols_datas_emp}

emprestimos_proc, rel_datas = fn.converter_tipos(
    emprestimos_proc,
    datas=datas_dict_emp,
)

# 4. Padronização de categóricas (excluindo estritamente as colunas temporais)
cols_cat_emp = [
    c for c in emprestimos_proc.select_dtypes(include=["object", "string"]).columns
    if c not in cols_datas_emp
]

emprestimos_proc, rel_cat_emp = fn.padronizar_categoricas(
    emprestimos_proc, colunas=cols_cat_emp
)

# 5. Otimização e coerção correta de tipos
emprestimos_proc["id_contrato"] = emprestimos_proc["id_contrato"].astype("int64")
emprestimos_proc["id_cliente"] = emprestimos_proc["id_cliente"].astype("int64")

if "qtd_parcelas_planejadas" in emprestimos_proc.columns:
    emprestimos_proc["qtd_parcelas_planejadas"] = emprestimos_proc["qtd_parcelas_planejadas"].astype("Int64")

if "indicador_seguro_contratado" in emprestimos_proc.columns:
    emprestimos_proc["indicador_seguro_contratado"] = emprestimos_proc["indicador_seguro_contratado"].astype("Int8")

for col in cols_cat_emp:
    emprestimos_proc[col] = emprestimos_proc[col].astype("category")

emprestimos_tratada = emprestimos_proc.copy()

# 6. Asserções de integridade e schema
assert len(emprestimos_tratada) == n_linhas_esperadas_emp, (
    f"Perda de registros: esperado {n_linhas_esperadas_emp}, obtido {len(emprestimos_tratada)}"
)
assert emprestimos_tratada["id_contrato"].nunique() == n_contratos_esperados_emp, (
    f"Inconsistência de chave: esperado {n_contratos_esperados_emp} contratos únicos, "
    f"obtido {emprestimos_tratada['id_contrato'].nunique()}"
)
assert "valor_credito_concedido" in emprestimos_tratada.columns, "Coluna valor_credito_concedido ausente"
assert "valor_credito" in emprestimos_tratada.columns, "Coluna valor_credito ausente"

for c_dt in cols_datas_emp:
    assert pd.api.types.is_datetime64_any_dtype(emprestimos_tratada[c_dt]), (
        f"Falha estrutural: {c_dt} deveria ser datetime64, mas está como {emprestimos_tratada[c_dt].dtype}"
    )

print("--- Base Empréstimos Tratada com Sucesso ---")
print(f"Shape final: {emprestimos_tratada.shape}")
print(f"Contratos únicos: {emprestimos_tratada['id_contrato'].nunique()}")
print(f"Clientes distintos: {emprestimos_tratada['id_cliente'].nunique()}")

In [ ]:
emprestimos_tratada.info()

In [ ]:
emprestimos_tratada.head()

In [ ]:
# --- TRATAMENTO E TIPAGEM: HISTÓRICO DE PARCELAS ---

# 1. Guarda referências dinâmicas para validação por invariantes
n_linhas_originais_parc = len(parcelas)
n_duplicadas_integrais_parc = parcelas.duplicated().sum()

# 2. Remoção de duplicatas estritas
parcelas_tratada = parcelas.drop_duplicates().copy()

# 3. Conversão de tipos estruturais e temporais
parcelas_tratada["id_contrato"] = parcelas_tratada["id_contrato"].astype("int64")
parcelas_tratada["id_cliente"] = parcelas_tratada["id_cliente"].astype("int64")
parcelas_tratada["versao_parcela"] = parcelas_tratada["versao_parcela"].astype("Int64")
parcelas_tratada["numero_parcela"] = parcelas_tratada["numero_parcela"].astype("Int64")

# Conversão explícita de datas
parcelas_tratada["data_prevista_pagamento"] = pd.to_datetime(
    parcelas_tratada["data_prevista_pagamento"], errors="coerce"
)
parcelas_tratada["data_real_pagamento"] = pd.to_datetime(
    parcelas_tratada["data_real_pagamento"], errors="coerce"
)

# Garantia de tipagem dos valores monetários
parcelas_tratada["valor_previsto_parcela"] = parcelas_tratada["valor_previsto_parcela"].astype("float64")
parcelas_tratada["valor_pago_parcela"] = parcelas_tratada["valor_pago_parcela"].astype("float64")

# 4. Asserções de integridade estrutural
assert len(parcelas_tratada) == (n_linhas_originais_parc - n_duplicadas_integrais_parc), (
    "Inconsistência volumétrica após remoção de duplicatas em parcelas"
)
assert not parcelas_tratada.duplicated().any(), "Ainda existem duplicatas integrais em parcelas_tratada"
assert pd.api.types.is_datetime64_any_dtype(parcelas_tratada["data_real_pagamento"]), (
    "A coluna 'data_real_pagamento' não foi convertida para datetime64"
)
assert pd.api.types.is_datetime64_any_dtype(parcelas_tratada["data_prevista_pagamento"]), (
    "A coluna 'data_prevista_pagamento' não foi convertida para datetime64"
)

print("--- Base Parcelas Tratada com Sucesso ---")
print(f"Linhas originais: {n_linhas_originais_parc}")
print(f"Duplicatas integrais removidas: {n_duplicadas_integrais_parc}")
print(f"Shape final: {parcelas_tratada.shape}")
print(f"Contratos distintos com histórico de parcelas: {parcelas_tratada['id_contrato'].nunique()}")

In [ ]:
parcelas_tratada.info()

In [ ]:
parcelas_tratada.head()

## Documentação de Decisões: Higienização e Tipagem das Bases Históricas

Esta etapa estabelece a camada de dados limpos (*Silver*) para o desenvolvimento do modelo, padronizando formatos, resolvendo inconsistências estruturais e garantindo integridade relacional sem vazamento de dados. A base de inferência (`submissao`) é preservada para avaliação posterior.

---

### 1. Base `cadastral` (`cadastral_tratada`)

- **Datas Sentinela e Inconsistentes:**
  - Identificação de anos de nascimento irrealistas (ano <= 1920).
  - Criação da flag booleana `data_nascimento_invalida` para rastreabilidade de negócio.
  - Conversão dos valores anômalos em `data_nascimento` para nulo (`pd.NaT`).
- **Conversão de Tipos Estruturais:**
  - `data_nascimento`: convertida para `datetime64[ns]` via múltiplos formatos (`%Y-%m-%d`, `%d/%m/%Y`).
  - `renda_anual`: higienização textual de moedas e conversão para `float64`.
  - `qtd_membros_familia`: convertida para inteiro com suporte a nulos (`Int64`).
- **Padronização Categórica e Placeholders:**
  - Remoção de acentuação, padronização em minúsculas e eliminação de espaços duplicados em todas as variáveis textuais.
  - Conversão de placeholders de ausência disfarçada (ex.: `"não informado"`, `"unknown"`, `"-"`) em `NaN` via `fn.padronizar_categoricas`.
- **Deduplicação e Atualização Cadastral:**
  - Clientes repetidos consolidados via `.combine_first()`.
  - A 2ª ocorrência (registro mais recente) prevalece como fonte primária; valores ausentes herdam os dados preenchidos na 1ª ocorrência.
- **Otimização de Memória e Conformidade:**
  - Conversão de `id_cliente` para `int64`, `qtd_filhos` para `Int64` e colunas textuais para `category`.
  - Remoção da coluna sensível `sexo` da base de modelagem para assegurar equidade algorítmica (*fairness*).
- **Validação de Integridade (Asserções Dinâmicas):**
  - Relação 1:1 estrita: total de linhas final igual ao número de `id_cliente` únicos originais (50.000).

---

### 2. Base `emprestimos` (`emprestimos_tratada`)

- **Preservação Volumétrica e Integridade Histórica:**
  - Manutenção de todos os contratos e histórico original (211.944 linhas), sem exclusão de linhas nem engenharia antecipada.
- **Conversão Temporal:**
  - `data_decisao` e `data_primeiro_vencimento`: convertidas para `datetime64[ns]`.
- **Padronização Categórica:**
  - Higienização textual e conversão de placeholders em `NaN` em atributos como `tipo_contrato`, `status_contrato`, etc.
- **Otimização de Memória e Tipagem:**
  - `id_contrato` e `id_cliente`: tipificados como `int64`.
  - Colunas categóricas textuais convertidas para `category`.
- **Validação de Integridade (Asserções Dinâmicas):**
  - Conservação exata da volumetria de entrada (`len(emprestimos_tratada) == len(emprestimos)`).
  - Unicidade estrita da chave primária `id_contrato` comprovada via assert.

---

### 3. Base `parcelas` (`parcelas_tratada`)

- **Eliminação de Duplicatas Técnicas:**
  - Remoção estrita de duplicatas integrais exatas (`drop_duplicates()`), preservando pagamentos parciais e amortizações legítimas.
- **Conversão Temporal:**
  - `data_prevista_pagamento` e `data_pagamento`: padronizadas para `datetime64[ns]`.
- **Otimização Estrutural e Tipagem:**
  - `id_contrato`: tipificado como `int64`.
  - `versao_parcela` e `numero_parcela`: convertidos para inteiros anuláveis (`Int64`).
- **Validação de Integridade (Asserções Dinâmicas):**
  - Conservação volumétrica exata calculada dinamicamente: `n_linhas_originais - n_duplicadas_integrais`.
  - Assert garantindo zero duplicatas integrais remanescentes na saída.

---

### Resumo Comparativo de Contratos de Saída

| Base | Chave Primária | Relação | Linhas Originais | Linhas Finais | Ação Principal de Limpeza |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **`cadastral_tratada`** | `id_cliente` | 1 : 1 | 50.600 | 50.000 | Consolidação por recência + fallback; exclusão de `sexo`; flag de nascimento. |
| **`emprestimos_tratada`** | `id_contrato` | 1 : N (cliente) | 211.944 | 211.944 | Conversão de datas e categorias; conservação estrita de contratos. |
| **`parcelas_tratada`** | Composta (`id_contrato`, `versao`, `num`) | 1 : M (contrato) | 1.664.384 | 1.659.408 | Exclusão de 4.976 duplicatas integrais; tipagem temporal e de contadores. |

## Rollup de parcelas por contrato e criação da target

### OVER60MOB12

In [ ]:
# --- ROLLUP DE PARCELAS POR CONTRATO E TARGET OVER60MOB12 ---

# 1. Parâmetros e regras de negócio documentadas
tolerancia_centavos = 0.01
limite_dias_atraso = 60
dias_mes_mob = 30
multiplicador_mob = 12
dias_janela_mob = dias_mes_mob * multiplicador_mob  # 360 dias

# Referências estruturais da base
n_contratos_esperados_parc = parcelas_tratada["id_contrato"].nunique()
data_corte_sistema = parcelas_tratada["data_real_pagamento"].max()

# 2. Seleciona a versão ativa mais recente de cada prestação (renegociações)
max_versoes = (
    parcelas_tratada.groupby(["id_contrato", "numero_parcela"])["versao_parcela"]
    .transform("max")
)
parc_ativas = parcelas_tratada.loc[parcelas_tratada["versao_parcela"] == max_versoes].copy()

# 3. Consolidação financeira no grão (id_contrato, numero_parcela)
resumo_parcelas = parc_ativas.groupby(["id_contrato", "numero_parcela"]).agg(
    data_prevista_pagamento=("data_prevista_pagamento", "first"),
    valor_previsto_parcela=("valor_previsto_parcela", "first"),
    total_pago=("valor_pago_parcela", "sum"),
    ultima_data_pagamento=("data_real_pagamento", "max")
).reset_index()

# Quitação financeira via tolerância
resumo_parcelas["quitada"] = (
    resumo_parcelas["total_pago"] >= (resumo_parcelas["valor_previsto_parcela"] - tolerancia_centavos)
)

# Identifica pagamentos com valor positivo sem data de registro
sem_data = (
    parc_ativas["valor_pago_parcela"].gt(0)
    & parc_ativas["data_real_pagamento"].isna()
).groupby([parc_ativas["id_contrato"], parc_ativas["numero_parcela"]]).any().reset_index(drop=True)

resumo_parcelas["pagamento_sem_data"] = sem_data

incerto = (
    resumo_parcelas["pagamento_sem_data"]
    | ~resumo_parcelas["valor_previsto_parcela"].gt(0)
    | resumo_parcelas["data_prevista_pagamento"].isna()
)

# 4. Apuração da data de liquidação efetiva e dias de atraso (DPD)
data_liquidacao = np.where(
    resumo_parcelas["quitada"] & resumo_parcelas["ultima_data_pagamento"].notna(),
    resumo_parcelas["ultima_data_pagamento"],
    np.where(
        resumo_parcelas["quitada"],
        resumo_parcelas["data_prevista_pagamento"],
        data_corte_sistema
    )
)
resumo_parcelas["data_liquidacao"] = pd.to_datetime(data_liquidacao)

dias_diferenca = (
    resumo_parcelas["data_liquidacao"] - resumo_parcelas["data_prevista_pagamento"]
).dt.days

resumo_parcelas["dias_atraso"] = (
    dias_diferenca.clip(lower=0)
    .astype("Int64")
    .mask(incerto)
)

# 5. Vinculação com marco de origem do crédito (sem persistir na tabela final)
col_origem = [c for c in ["data_liberacao", "data_decisao"] if c in emprestimos_tratada.columns][0]
origem_contrato = (
    emprestimos_tratada[["id_contrato", col_origem]]
    .drop_duplicates("id_contrato")
    .rename(columns={col_origem: "_data_origem"})
)

base_mob = resumo_parcelas.merge(origem_contrato, on="id_contrato", how="left")
menor_venc = base_mob.groupby("id_contrato")["data_prevista_pagamento"].transform("min")
base_mob["_data_origem"] = base_mob["_data_origem"].fillna(menor_venc - pd.Timedelta(days=dias_mes_mob))

# 6. Avaliação temporal da janela MOB 12 (360 dias)
data_limiar_parcela = base_mob["data_prevista_pagamento"] + pd.Timedelta(days=limite_dias_atraso)
fim_janela_prestacao = base_mob["_data_origem"] + pd.Timedelta(days=dias_janela_mob)

atingiu_limite = base_mob["dias_atraso"].ge(limite_dias_atraso)

evento_no_mob = (
    atingiu_limite
    & (data_limiar_parcela <= fim_janela_prestacao)
)

incerta_no_mob = (
    base_mob["dias_atraso"].isna()
    & (data_limiar_parcela <= fim_janela_prestacao)
)

# 7. Agrupamento no grão de contrato
contratos_unicos = (
    base_mob[["id_contrato", "_data_origem"]]
    .drop_duplicates("id_contrato")
    .set_index("id_contrato")
)

teve_evento = evento_no_mob.groupby(base_mob["id_contrato"]).any()
teve_incerteza = incerta_no_mob.groupby(base_mob["id_contrato"]).any()

fim_janela_contrato = contratos_unicos["_data_origem"] + pd.Timedelta(days=dias_janela_mob)
janela_maturada = fim_janela_contrato <= data_corte_sistema

# Métricas comportamentais no horizonte MOB 12
prestacoes_no_mob = base_mob.loc[base_mob["data_prevista_pagamento"] <= fim_janela_prestacao]
max_dpd_mob12 = prestacoes_no_mob.groupby("id_contrato")["dias_atraso"].max()
qtd_prestacoes_mob12 = prestacoes_no_mob.groupby("id_contrato")["numero_parcela"].count()

# Montagem do DataFrame final parcelas_por_contrato
parcelas_por_contrato = pd.DataFrame(index=contratos_unicos.index)
parcelas_por_contrato["max_dias_atraso_mob12"] = max_dpd_mob12.reindex(contratos_unicos.index)
parcelas_por_contrato["qtd_parcelas_mob12"] = qtd_prestacoes_mob12.reindex(contratos_unicos.index).fillna(0).astype("Int64")
parcelas_por_contrato["contrato_maduro_mob12"] = janela_maturada

target_serie = pd.Series(pd.NA, index=contratos_unicos.index, dtype="Int8")
target_serie = target_serie.where(~(janela_maturada & ~teve_incerteza), 0)
target_serie = target_serie.where(~teve_evento, 1)

parcelas_por_contrato["OVER60MOB12"] = target_serie
parcelas_por_contrato = parcelas_por_contrato.reset_index()

# 8. Asserções de integridade
assert len(parcelas_por_contrato) == n_contratos_esperados_parc, (
    f"Inconsistência volumétrica: esperado {n_contratos_esperados_parc}, obtido {len(parcelas_por_contrato)}"
)
assert parcelas_por_contrato["id_contrato"].nunique() == len(parcelas_por_contrato), (
    "Falha de unicidade em id_contrato"
)
assert "_data_origem" not in parcelas_por_contrato.columns, (
    "A coluna _data_origem não deve ser persistida na tabela de parcelas"
)

# 9. Relatório de validação
dist_target = parcelas_por_contrato["OVER60MOB12"].value_counts(dropna=False)
total_avaliados = dist_target.get(0, 0) + dist_target.get(1, 0)
taxa_inadimplencia = (dist_target.get(1, 0) / total_avaliados) * 100

print("--- Rollup de Parcelas por Contrato Concluído ---")
print(f"Total de contratos consolidados: {len(parcelas_por_contrato)}")
print(f"Bons (0): {dist_target.get(0, 0)}")
print(f"Maus (1): {dist_target.get(1, 0)}")
print(f"Indeterminados (<NA>): {dist_target.get(pd.NA, 0)}")
print(f"Taxa de Inadimplência observada: {taxa_inadimplencia:.4f}%")

In [ ]:
# --- ROLLUP DE PARCELAS E TARGET OVER60MOB12: VARIANTES DENTRO E FORA do período 10+ meses---

# 1. Parâmetros
tolerancia_centavos = 0.01
limite_dias_atraso = 60
dias_mes_mob = 30
multiplicador_mob = 12
dias_janela_mob = dias_mes_mob * multiplicador_mob  # 360 dias
filtrar_versao_maxima = True  # ajustar conforme o resultado do diagnóstico de versões

chave_parcela = ["id_contrato", "numero_parcela"]
n_contratos_esperados_parc = parcelas_tratada["id_contrato"].nunique()
data_corte_sistema = parcelas_tratada["data_real_pagamento"].max()

# 2. Seleção de versão (parametrizada)
if filtrar_versao_maxima:
    versao_max = parcelas_tratada.groupby(chave_parcela)["versao_parcela"].transform("max")
    parc_base = parcelas_tratada.loc[parcelas_tratada["versao_parcela"] == versao_max].copy()
else:
    parc_base = parcelas_tratada.copy()

# Pagamento positivo sem data (incerteza), marcado por linha antes de agregar
parc_base["_pgto_sem_data"] = (
    parc_base["valor_pago_parcela"].gt(0) & parc_base["data_real_pagamento"].isna()
)

# 3. Consolidação no grão (id_contrato, numero_parcela)
resumo_parcelas = (
    parc_base.groupby(chave_parcela)
    .agg(
        data_prevista_pagamento=("data_prevista_pagamento", "first"),
        valor_previsto_parcela=("valor_previsto_parcela", "first"),
        total_pago=("valor_pago_parcela", "sum"),
        ultima_data_pagamento=("data_real_pagamento", "max"),
        pagamento_sem_data=("_pgto_sem_data", "any"),
    )
    .reset_index()
)

# 4. Quitação e incerteza
resumo_parcelas["quitada"] = (
    resumo_parcelas["total_pago"]
    >= (resumo_parcelas["valor_previsto_parcela"] - tolerancia_centavos)
)
incerto = (
    resumo_parcelas["pagamento_sem_data"]
    | ~resumo_parcelas["valor_previsto_parcela"].gt(0)
    | resumo_parcelas["data_prevista_pagamento"].isna()
)

# 5. Data de liquidação e dias de atraso
data_liquidacao = (
    resumo_parcelas["ultima_data_pagamento"].where(resumo_parcelas["quitada"])
    .fillna(resumo_parcelas["data_prevista_pagamento"].where(resumo_parcelas["quitada"]))
    .fillna(data_corte_sistema)
)
dias_diferenca = (data_liquidacao - resumo_parcelas["data_prevista_pagamento"]).dt.days
resumo_parcelas["dias_atraso"] = dias_diferenca.clip(lower=0).astype("Int64").mask(incerto)

# 6. Origem do contrato: data_decisao (intermediária, não persistida)
origem_por_contrato = emprestimos_tratada.set_index("id_contrato")["data_decisao"]
resumo_parcelas["_data_origem"] = resumo_parcelas["id_contrato"].map(origem_por_contrato)
assert resumo_parcelas["_data_origem"].notna().all(), (
    "Há contratos de parcelas sem data_decisao em emprestimos_tratada"
)

# 7. Eventos dentro da janela MOB 12
limiar_dentro_janela = (
    resumo_parcelas["data_prevista_pagamento"] + pd.Timedelta(days=limite_dias_atraso)
) <= (resumo_parcelas["_data_origem"] + pd.Timedelta(days=dias_janela_mob))

evento_no_mob = resumo_parcelas["dias_atraso"].ge(limite_dias_atraso) & limiar_dentro_janela
incerta_no_mob = resumo_parcelas["dias_atraso"].isna() & limiar_dentro_janela

# 8. Agregação por contrato
id_contrato_parcela = resumo_parcelas["id_contrato"]
teve_evento = evento_no_mob.groupby(id_contrato_parcela).any()
teve_incerteza = incerta_no_mob.groupby(id_contrato_parcela).any()

origem_contrato = resumo_parcelas.groupby("id_contrato")["_data_origem"].first()
janela_maturada = (origem_contrato + pd.Timedelta(days=dias_janela_mob)) <= data_corte_sistema

# 9. Variante DENTRO: positivo assim que o evento ocorre, mesmo em janela imatura
target_dentro = pd.Series(pd.NA, index=origem_contrato.index, dtype="Int8")
target_dentro = target_dentro.mask(janela_maturada & ~teve_incerteza, 0)
target_dentro = target_dentro.mask(teve_evento, 1)

# 10. Variante FORA: positivos de janela imatura ficam <NA>
target_fora = pd.Series(pd.NA, index=origem_contrato.index, dtype="Int8")
target_fora = target_fora.mask(janela_maturada & ~teve_incerteza, 0)
target_fora = target_fora.mask(janela_maturada & teve_evento, 1)

# 11. Tabela de contratos: apenas chave e targets candidatos
parcelas_por_contrato = pd.DataFrame({
    "OVER60MOB12_dentro": target_dentro,
    "OVER60MOB12_fora": target_fora,
}).reset_index()

# 12. Asserções
assert len(parcelas_por_contrato) == n_contratos_esperados_parc, (
    f"Esperado {n_contratos_esperados_parc} contratos, obtido {len(parcelas_por_contrato)}"
)
assert parcelas_por_contrato["id_contrato"].is_unique, "Falha de unicidade em id_contrato"
assert not any(c.startswith("_") for c in parcelas_por_contrato.columns), "Coluna intermediária vazou"

# 13. Comparativo das variantes
linhas = []
for nome, serie in (("dentro", target_dentro), ("fora", target_fora)):
    n_bons = int(serie.eq(0).sum())
    n_maus = int(serie.eq(1).sum())
    linhas.append({
        "variante": nome,
        "bons": n_bons,
        "maus": n_maus,
        "indeterminados": int(serie.isna().sum()),
        "avaliaveis": n_bons + n_maus,
        "taxa_maus_%": n_maus / (n_bons + n_maus) * 100,
    })

print(f"Contratos consolidados: {len(parcelas_por_contrato)}")
print(f"Contratos com janela madura: {int(janela_maturada.sum())}")
print(f"Positivos em janela imatura (removidos na variante 'fora'): {int((teve_evento & ~janela_maturada).sum())}")
display(pd.DataFrame(linhas).set_index("variante"))

In [ ]:
# --- TAXA DE INADIMPLÊNCIA MENSAL (VARIANTE "DENTRO") POR MÊS DE data_decisao ---
import matplotlib.pyplot as plt

# 1. Base no grão de contrato
base_mensal = pd.DataFrame({
    "mes": origem_contrato.dt.to_period("M"),
    "target": target_dentro,
    "maduro": janela_maturada,
})

# 2. Agregação mensal (a soma ignora <NA>, count conta apenas avaliáveis)
mensal = base_mensal.groupby("mes").agg(
    contratos=("target", "size"),
    avaliaveis=("target", "count"),
    maus=("target", "sum"),
    pct_maduros=("maduro", "mean"),
)
mensal["taxa_maus_%"] = mensal["maus"] / mensal["avaliaveis"].where(mensal["avaliaveis"] > 0) * 100
display(mensal)

# 3. Gráfico: taxa (linha) e volume avaliável (barras)
x = mensal.index.to_timestamp()
fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.bar(x, mensal["avaliaveis"], width=20, color="lightgray")
ax1.set_ylabel("Contratos avaliáveis")
ax2 = ax1.twinx()
ax2.plot(x, mensal["taxa_maus_%"], color="crimson", marker="o")
ax2.set_ylabel("Taxa de maus (%)")
ax1.set_title("OVER60MOB12 (dentro): taxa por mês de data_decisao")
plt.tight_layout()
plt.show()

In [ ]:
## Rollup de parcelas por contrato e criação da target
